# ALPHASE v0.2 — Colab GPU
이 노트북은 작성·구문 검사 완료 상태이며 Colab 브라우저에서 실제 실행은 아직 검증하지 않았습니다. GPU 런타임을 선택하세요.

In [ ]:
import torch
print(torch.__version__,torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU 런타임이 필요합니다"
print(torch.cuda.get_device_name(0))

## 업로드
ALPHASE_code.zip, train.zip, valid.zip, test.zip을 업로드하세요.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
files.upload()
target=Path("/content").resolve()
with zipfile.ZipFile("/content/ALPHASE_code.zip") as z:
    for n in z.namelist():
        assert (target/n).resolve().is_relative_to(target)
    z.extractall(target)
%cd /content/ALPHASE
!python -m pip install -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUNS=Path("/content/drive/MyDrive/ALPHASE_runs_v02")
RUNS.mkdir(exist_ok=True)

In [ ]:
!python -m alphase prepare --archives /content --data /content/data
!python -m alphase inspect --data /content/data --official-counts --output "{RUNS}/data_report.json"
!python -m unittest discover -s tests -v

## GPU 최소 실행
한 배치 학습 뒤 전체 검증 이미지를 원본 해상도로 평가합니다. 실제 성능 실험이 아닙니다.

In [ ]:
import json
cfg=json.loads(Path("configs/core.json").read_text())
smoke=cfg.copy();smoke.update(device="cuda",epochs=1,batch_size=2)
Path("configs/colab_smoke.json").write_text(json.dumps(smoke))
!python -m alphase train --data /content/data --config configs/colab_smoke.json --output "{RUNS}/gpu_smoke" --steps 1

## Core 학습
사전학습은 규정 확인 전 false를 유지합니다. 설정 자동 탐색은 별도 tune 명령으로 수행할 수 있습니다.

In [ ]:
cfg["device"]="cuda"
cfg["pretrained"]=False
Path("configs/colab_core.json").write_text(json.dumps(cfg))
!python -m alphase train --data /content/data --config configs/colab_core.json --output "{RUNS}/core"

중단 후에는 같은 코드·데이터·config를 준비한 뒤 다음 명령으로 재개하세요.
```python
!python -m alphase train --data /content/data --config configs/colab_core.json --output "{RUNS}/core" --resume "{RUNS}/core/last.pt"
```

In [ ]:
!python -m alphase evaluate --data /content/data --checkpoint "{RUNS}/core/best.pt" --device cuda --output "{RUNS}/core/evaluation" --cache
!python -m alphase infer --data /content/data --checkpoint "{RUNS}/core/inference.pt" --device cuda --output "{RUNS}/predictions" --contract configs/submission_contract.json
!python -m alphase validate-output --data /content/data --predictions "{RUNS}/predictions" --contract configs/submission_contract.json --output "{RUNS}/output_check.json"

In [ ]:
import shutil
shutil.make_archive("/content/ALPHASE_predictions","zip",str(RUNS/"predictions"))
files.download("/content/ALPHASE_predictions.zip")

## 추가 실험
전체 A–G 자동 실행은 많은 GPU 시간을 사용합니다. README의 experiment-plan/experiments, ensemble-search, package 명령을 확인하세요. 계약 미확정 상태에서는 최종 제출 패키징이 중단됩니다.